# Assignment: Build an AI Student Helpdesk Assistant



## Business Scenario
Northstar Learning Hub has a handbook containing its student policies. The academic team wants a basic AI assistant that can answer learners' questions **using the handbook**, instead of guessing.

You will build a small pipeline that:
1. Reads the provided PDF and divides it into text chunks.
2. Tests **zero-shot and few-shot prompts** to categorize student questions.
3. Calls the **DeepSeek API** from Python.
4. Retrieves a relevant handbook chunk using **keyword overlap**.
5. Uses a **LangChain chain** to generate a grounded answer.
6. Tests whether answers are supported by the PDF, including for missing information.


## Learning Outcomes


By completing this assignment, you should be able to:
- Apply system/user messages and zero-shot/few-shot prompting.
- Send API requests using `requests` with configurable `temperature` and `max_tokens`.
- Extract and chunk PDF text with Python and `pypdf`.
- Retrieve a relevant chunk with simple word-overlap scoring.
- Connect `PromptTemplate`, `RunnableLambda`, and `StrOutputParser`.
- Evaluate grounded answers and identify unsupported responses.



## Provided resources




- [Northstar_Student_Handbook.pdf](https://drive.google.com/file/d/13GKO3S8Z0ek6nOHqZeizAbqu3X9IDcsi/view?usp=drive_link)

- Example test questions near the end


## Submission



Upload your completed `.ipynb` with executed outputs on GitHub and share the link of repository.

## Part A — Prepare Your Colab Environment
### Step 0. Install libraries and load your API key

Run the setup cells below. The installer and key-loading code are provided because the assessed work begins with the functions in Step 1.

1. In Colab, open the **Secrets** sidebar.
2. Create a secret named `DEEPSEEK_API_KEY` and allow this notebook to use it.
3. Never paste your secret into a code cell or output.
4. Use a DeepSeek model available to your account. The default shown below is `deepseek-chat`; update it only if your account requires a different supported model.

**Purpose of Code:** This setup cell installs the required libraries, imports the tools used throughout the assignment, and loads your DeepSeek API key securely. Later functions depend on these imports and settings to run.


In [1]:
%pip -q install requests pypdf langchain-core

import os
import re
import requests
from typing import Any
from pypdf import PdfReader
from google.colab import files, userdata
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser

DEEPSEEK_API_KEY = userdata.get("DEEPSEEK_API_KEY")
if not DEEPSEEK_API_KEY:
    raise ValueError("Save DEEPSEEK_API_KEY in Colab Secrets and enable access.")

DEEPSEEK_API_URL = "https://api.deepseek.com/chat/completions"
DEEPSEEK_MODEL = "deepseek-chat"
HEADERS = {
    "Authorization": f"Bearer {DEEPSEEK_API_KEY}",
    "Content-Type": "application/json",
}
print("Setup complete. API key loaded securely.")

Setup complete. API key loaded securely.


### Step 1. Upload the student handbook PDF

Download the **provided** handbook PDF and upload it in the next cell. Run this once each time the Colab runtime is reset. We use the file's original name in all later steps.

**Purpose of Code:** This cell makes the handbook PDF available inside the Colab runtime and saves its file path. The PDF-reading function in the next step will use this path to open the correct document.


In [3]:
uploaded = files.upload()
if not uploaded:
    raise ValueError("Please upload the supplied student handbook PDF.")

PDF_PATH = next(iter(uploaded))
if not PDF_PATH.lower().endswith(".pdf"):
    raise ValueError("The uploaded file must be a PDF.")
print("Uploaded:", PDF_PATH)

Saving Northstar_Student_Handbook.pdf to Northstar_Student_Handbook.pdf
Uploaded: Northstar_Student_Handbook.pdf


### Step 2. Extract text from the PDF

**Your task:** Complete `read_handbook_pdf(pdf_path)`.

- Open the file with `PdfReader`.
- Extract the text from every page.
- Ignore any empty page text.
- Join the extracted text into **one string**.
- Return the combined string.

**Expected input:** Path to one text-based PDF.  
**Expected output:** A non-empty string containing handbook text.

**Hint:** Recall `page.extract_text()` from the LangChain & RAG lesson. Scanned/image-only PDFs are out of scope.

**Purpose of Code:** This function converts the uploaded handbook from a PDF into plain text. The assistant needs this text so it can search the handbook and use its policies when answering student questions.


In [4]:
def read_handbook_pdf(pdf_path: str) -> str:
    """Return readable text extracted from all pages of the handbook PDF."""
    # TODO 1: Open the PDF using PdfReader.
    # TODO 2: Extract and collect text from each page.
    # TODO 3: Combine the page text and return it.
    # ----------------------DONE----------------------

    """
    Extract readable text from every page of the handbook PDF.

    Parameters:
        pdf_path (str): Path to the uploaded PDF file.

    Returns:
        str: Combined text extracted from all non-empty PDF pages.
    """

    reader = PdfReader(pdf_path)

    extracted_pages = []

    for page_number, page in enumerate(reader.pages, start=1):
        page_text = page.extract_text()

        if page_text and page_text.strip():
            extracted_pages.append(page_text.strip())

    handbook_text = "\n\n".join(extracted_pages)

    if not handbook_text.strip():
        raise ValueError(
            "No readable text was extracted from the PDF. "
            "Please check whether the PDF contains selectable text."
        )

    return handbook_text

In [5]:
# Extract text from the uploaded student handbook
handbook_text = read_handbook_pdf(PDF_PATH)

# Display extraction statistics
print("PDF Extraction Successful!")
print("-" * 50)

print("PDF File:", PDF_PATH)
print("Total Pages:", len(PdfReader(PDF_PATH).pages))
print("Extracted Characters:", len(handbook_text))
print("Extracted Words:", len(handbook_text.split()))

# Display a preview of the extracted text
print("\nHANDBOOK TEXT PREVIEW")
print("-" * 50)

print(handbook_text[:2000])

PDF Extraction Successful!
--------------------------------------------------
PDF File: Northstar_Student_Handbook.pdf
Total Pages: 2
Extracted Characters: 3085
Extracted Words: 461

HANDBOOK TEXT PREVIEW
--------------------------------------------------
FICTIONAL TRAINING DOCUMENT | GenAI Foundations Assignment
Page 1
Northstar Learning Hub
Student Handbook | Version 1.0 | Fictional policy document for classroom assignments
1. Program Overview
Northstar Learning Hub provides an eight-week, instructor-led online training program. All dates and
deadlines follow India Standard Time (IST). Learners should read announcements in the learning portal
every week.
2. Enrollment and Orientation
New learners receive their portal login and an orientation email within two business days after
enrollment confirmation. Orientation includes course navigation and how to contact the support team.
3. Class Attendance
Live sessions take place on weekdays according to the cohort timetable. To qualify for a

### Step 3. Split the document into chunks

**Your task:** Complete `split_text_into_chunks(text, chunk_size, chunk_overlap)`.

- Split the text into smaller, overlapping character-based chunks.
- Start from character index `0` and move by `chunk_size - chunk_overlap`.
- Remove empty chunks and return a list of strings.
- Check that overlap is smaller than chunk size.

**Expected output:** A list such as `['first chunk ...', 'second chunk ...']`.

**Hint:** The RAG lesson used a simple `start`, `end`, and `while` loop. Do not use a new text-splitting library.

**Purpose of Code:** This function breaks the long handbook text into smaller, overlapping sections. Smaller chunks make it easier to find relevant policy information without sending the entire handbook to the LLM.


In [6]:
def split_text_into_chunks(text: str, chunk_size: int = 1000, chunk_overlap: int = 150) -> list[str]:
    """Return overlapping character-based chunks from the PDF text."""
    # TODO 1: Validate that 0 <= chunk_overlap < chunk_size.
    # TODO 2: Slice the text using a moving start and end position.
    # TODO 3: Collect non-empty chunks, then return them.
    # ----------------------DONE----------------------

    """
    Split the extracted handbook text into overlapping
    character-based chunks.

    Parameters:
        text (str): Complete extracted PDF text.
        chunk_size (int): Maximum characters per chunk.
        chunk_overlap (int): Number of overlapping characters
                             between consecutive chunks.

    Returns:
        list[str]: A list of non-empty text chunks.
    """

    if chunk_size <= 0:
        raise ValueError("chunk_size must be greater than zero.")

    if not 0 <= chunk_overlap < chunk_size:
        raise ValueError(
            "chunk_overlap must be non-negative "
            "and smaller than chunk_size."
        )

    chunks = []
    step = chunk_size - chunk_overlap
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]

        if chunk.strip():
            chunks.append(chunk)

        start += step

    return chunks

In [7]:
# Generate chunks from the extracted handbook text
handbook_chunks = split_text_into_chunks(
    handbook_text,
    chunk_size=1000,
    chunk_overlap=150
)

# Display chunking statistics
print("Text Chunking Successful!")
print("-" * 50)

print("Total Extracted Characters:", len(handbook_text))
print("Total Number of Chunks:", len(handbook_chunks))
print("Chunk Size:", 1000)
print("Chunk Overlap:", 150)
print("Step Size:", 850)

# Display the first chunk
print("\nFIRST CHUNK PREVIEW")
print("-" * 50)
print(handbook_chunks[0])

# Display the second chunk
if len(handbook_chunks) > 1:
    print("\nSECOND CHUNK PREVIEW")
    print("-" * 50)
    print(handbook_chunks[1])

Text Chunking Successful!
--------------------------------------------------
Total Extracted Characters: 3085
Total Number of Chunks: 4
Chunk Size: 1000
Chunk Overlap: 150
Step Size: 850

FIRST CHUNK PREVIEW
--------------------------------------------------
FICTIONAL TRAINING DOCUMENT | GenAI Foundations Assignment
Page 1
Northstar Learning Hub
Student Handbook | Version 1.0 | Fictional policy document for classroom assignments
1. Program Overview
Northstar Learning Hub provides an eight-week, instructor-led online training program. All dates and
deadlines follow India Standard Time (IST). Learners should read announcements in the learning portal
every week.
2. Enrollment and Orientation
New learners receive their portal login and an orientation email within two business days after
enrollment confirmation. Orientation includes course navigation and how to contact the support team.
3. Class Attendance
Live sessions take place on weekdays according to the cohort timetable. To qualify fo

In [8]:
# Validate chunking output
assert isinstance(handbook_chunks, list)
assert len(handbook_chunks) > 0

# Check that all chunks contain text
assert all(chunk.strip() for chunk in handbook_chunks)

# Check maximum chunk length
assert all(len(chunk) <= 1000 for chunk in handbook_chunks)

# Verify overlap between the first two chunks
if len(handbook_chunks) > 1:
    overlap_found = (
        handbook_chunks[0][-150:] in handbook_chunks[1]
    )
    print("Overlap Verification:", overlap_found)

print("\nAll chunking validations passed!")

Overlap Verification: True

All chunking validations passed!


## Part B — Apply Prompt Engineering
### Step 4. Create a zero-shot question-classification prompt

**Your task:** Complete `build_zero_shot_prompt(question)`.

Write a prompt that asks the LLM to classify a question using **only** these categories:
`Enrollment`, `Assignments`, `Certificates`, `Support`, `Other`.

Requirements:
- Give a clear instruction but **no examples**.
- Include the student's question.
- Ask for **only the category name** in the response.

**Example input:** `"Can I resubmit my graded homework?"`  
**Expected category:** `Assignments`.

**Purpose of Code:** This function creates a zero-shot prompt that tells the LLM how to categorize a student question without examples. It provides a baseline for comparing different prompt-engineering techniques.


In [9]:
def build_zero_shot_prompt(question: str) -> str:
    """Return a zero-shot classification prompt for a student question."""
    # TODO 1: Explain the classification task and allowed labels.
    # TODO 2: Insert the student's question.
    # TODO 3: Specify the exact output format.
    # ----------------------DONE----------------------

    """
    Create a zero-shot prompt for classifying student questions.

    Parameters:
        question (str): Student's question.

    Returns:
        str: Classification prompt without examples.
    """


    prompt = f"""
        You are an AI Student Helpdesk Assistant for Northstar Learning Hub.

        Your task is to classify the student's question into exactly
        ONE of the following categories:

        1. Enrollment
        2. Assignments
        3. Certificates
        4. Support
        5. Other

        Classification Guidelines:
        - Enrollment: Questions related to admission, registration,
          joining the program, or enrollment procedures.
        - Assignments: Questions related to homework, assignment
          submission, deadlines, grading, or resubmission.
        - Certificates: Questions related to completion certificates,
          eligibility, or certificate issuance.
        - Support: Questions related to student assistance, mentoring,
          technical help, or guidance.
        - Other: Questions that do not belong to any of the above categories.

        Student Question:
        {question}

        Instructions:
        - Select only one category.
        - Use only the five categories specified above.
        - Do not provide explanations or additional information.
        - Return only the category name.
        """

    return prompt.strip()

In [10]:
# Example student question
sample_question = "Can I resubmit my graded homework?"

# Generate the zero-shot prompt
zero_shot_prompt = build_zero_shot_prompt(sample_question)

# Display the generated prompt
print("ZERO-SHOT CLASSIFICATION PROMPT")
print("=" * 60)
print(zero_shot_prompt)

ZERO-SHOT CLASSIFICATION PROMPT
You are an AI Student Helpdesk Assistant for Northstar Learning Hub.

        Your task is to classify the student's question into exactly
        ONE of the following categories:

        1. Enrollment
        2. Assignments
        3. Certificates
        4. Support
        5. Other

        Classification Guidelines:
        - Enrollment: Questions related to admission, registration,
          joining the program, or enrollment procedures.
        - Assignments: Questions related to homework, assignment
          submission, deadlines, grading, or resubmission.
        - Certificates: Questions related to completion certificates,
          eligibility, or certificate issuance.
        - Support: Questions related to student assistance, mentoring,
          technical help, or guidance.
        - Other: Questions that do not belong to any of the above categories.

        Student Question:
        Can I resubmit my graded homework?

        Instructions

In [11]:
# Validate the generated prompt

required_categories = [
    "Enrollment",
    "Assignments",
    "Certificates",
    "Support",
    "Other"
]

# Check whether all categories are present
for category in required_categories:
    assert category in zero_shot_prompt, (
        f"Missing category: {category}"
    )

# Check whether the question is included
assert sample_question in zero_shot_prompt

# Check that no examples have been inserted
assert "Example Question" not in zero_shot_prompt

print("Zero-shot prompt validation successful!")
print("All five categories are present.")
print("Student question is correctly included.")

Zero-shot prompt validation successful!
All five categories are present.
Student question is correctly included.


### Step 5. Create a few-shot question-classification prompt

**Your task:** Complete `build_few_shot_prompt(question)`.

Use the **same categories** and output format as Step 4, but add at least **three labeled examples** in the prompt.

You may use these examples:

| Example question | Category |
|---|---|
| How do I enroll in the next batch? | Enrollment |
| When is my assignment due? | Assignments |
| I cannot open my learning portal. | Support |

End with the new question and ask for its category. Return the complete prompt as a string.

**Purpose of Code:** This function creates a few-shot prompt with labeled examples of student questions. You will compare it with the zero-shot version to observe how examples can influence classification.


In [12]:
def build_few_shot_prompt(question: str) -> str:
    """Return a few-shot classification prompt with labeled examples."""
    # TODO 1: State the allowed categories.
    # TODO 2: Include three or more question/category examples.
    # TODO 3: Add the new question and request only the category.
    # ----------------------DONE----------------------


    """
    Create a few-shot classification prompt using
    labelled examples of student questions.

    Parameters:
        question (str): New student question to classify.

    Returns:
        str: Complete few-shot classification prompt.
    """

    prompt = f"""
        You are an AI Student Helpdesk Assistant for Northstar Learning Hub.

        Your task is to classify student questions into exactly
        ONE of the following five categories:

        1. Enrollment
        2. Assignments
        3. Certificates
        4. Support
        5. Other

        Study the following labelled examples carefully.

        EXAMPLE 1:
        Student Question: How do I enroll in the next batch?
        Category: Enrollment

        EXAMPLE 2:
        Student Question: When is my assignment due?
        Category: Assignments

        EXAMPLE 3:
        Student Question: I cannot open my learning portal.
        Category: Support

        Classification Guidelines:

        - Enrollment: Admission, registration, joining the program,
          or enrollment procedures.
        - Assignments: Homework, assignment submission, deadlines,
          grading, or resubmission.
        - Certificates: Completion certificates, eligibility,
          or certificate issuance.
        - Support: Student assistance, mentoring, technical help,
          or guidance.
        - Other: Questions that do not belong to the above categories.

        Now classify the following new student question.

        NEW STUDENT QUESTION:
        {question}

        Instructions:
        - Select exactly one category.
        - Follow the classification pattern demonstrated in the examples.
        - Use only the five permitted categories.
        - Do not provide explanations.
        - Return only the category name.
        """


    return prompt.strip()

In [12]:
# Define a sample student question
sample_question = "Can I resubmit my graded homework?"

# Generate the few-shot prompt
few_shot_prompt = build_few_shot_prompt(sample_question)

# Display the generated prompt
print("FEW-SHOT CLASSIFICATION PROMPT")
print("=" * 60)
print(few_shot_prompt)

FEW-SHOT CLASSIFICATION PROMPT
You are an AI Student Helpdesk Assistant for Northstar Learning Hub.

        Your task is to classify student questions into exactly
        ONE of the following five categories:

        1. Enrollment
        2. Assignments
        3. Certificates
        4. Support
        5. Other

        Study the following labelled examples carefully.

        EXAMPLE 1:
        Student Question: How do I enroll in the next batch?
        Category: Enrollment

        EXAMPLE 2:
        Student Question: When is my assignment due?
        Category: Assignments

        EXAMPLE 3:
        Student Question: I cannot open my learning portal.
        Category: Support

        Classification Guidelines:

        - Enrollment: Admission, registration, joining the program,
          or enrollment procedures.
        - Assignments: Homework, assignment submission, deadlines,
          grading, or resubmission.
        - Certificates: Completion certificates, eligibility,


In [13]:
# Validate the few-shot prompt

required_categories = [
    "Enrollment",
    "Assignments",
    "Certificates",
    "Support",
    "Other"
]

# Verify all permitted categories
for category in required_categories:
    assert category in few_shot_prompt

# Verify all three labelled examples
assert "How do I enroll in the next batch?" in few_shot_prompt
assert "When is my assignment due?" in few_shot_prompt
assert "I cannot open my learning portal." in few_shot_prompt

# Verify the new question
assert sample_question in few_shot_prompt

# Count the examples
example_count = few_shot_prompt.count("EXAMPLE")

assert example_count >= 3

print("Few-shot prompt validation successful!")
print("All five categories are present.")
print("Three labelled examples are present.")
print("New student question is correctly included.")

Few-shot prompt validation successful!
All five categories are present.
Three labelled examples are present.
New student question is correctly included.


## Part C — Call an LLM Using an API
### Step 6. Implement a reusable DeepSeek API function

**Your task:** Complete `call_deepseek(messages, temperature, max_tokens)` using **Python `requests`**, as in the LLM API lesson.

- Prepare the JSON request body with `model`, `messages`, `temperature`, and `max_tokens`.
- Send the request with `requests.post` to `DEEPSEEK_API_URL` using `HEADERS` and `timeout=60`.
- Call `raise_for_status()`.
- Parse the response JSON and return the assistant's message content as a string.
- Include any model-specific settings only if needed by your configured DeepSeek model.

**Expected input:** A list of `system` and `user` messages.  
**Expected output:** Generated text.

**Security:** Keep the API key in Colab Secrets; do not print it.

**Purpose of Code:** This function sends messages to the DeepSeek model and retrieves its response. Making it reusable allows both the classification tasks and the RAG pipeline to call the same API without duplicating request code.


In [13]:
def call_deepseek(
    messages: list[dict],
    temperature: float = 0.2,
    max_tokens: int = 250,
) -> str:
    """Send messages to DeepSeek and return generated assistant text."""
    # TODO 1: Build the JSON payload.
    # TODO 2: Send an HTTP POST with requests.
    # TODO 3: Check status, decode JSON, return the response text.
    # ----------------------DONE----------------------


    """
    Send messages to the DeepSeek API and return
    the generated assistant response.

    Parameters:
        messages (list[dict]): Conversation messages containing
                               system and user instructions.
        temperature (float): Controls randomness in the response.
        max_tokens (int): Maximum number of tokens to generate.

    Returns:
        str: Generated text from the DeepSeek assistant.
    """

    payload = {
        "model": DEEPSEEK_MODEL,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens
    }

    response = requests.post(
        DEEPSEEK_API_URL,
        headers=HEADERS,
        json=payload,
        timeout=60
    )

    response.raise_for_status()
    response_data = response.json()

    try:
        assistant_response = (
            response_data["choices"][0]["message"]["content"]
        )
    except (KeyError, IndexError, TypeError) as error:
        raise RuntimeError(
            f"Unexpected DeepSeek API response format: {response_data}"
        ) from error

    if not isinstance(assistant_response, str):
        raise RuntimeError(
            "DeepSeek returned an empty or non-text response."
        )

    return assistant_response.strip()

In [14]:
test_messages = [
    {
        "role": "system",
        "content": (
            "You are a student helpdesk classification assistant. "
            "Return only the appropriate category."
        )
    },
    {
        "role": "user",
        "content": build_zero_shot_prompt(
            "Can I resubmit my graded homework?"
        )
    }
]

test_response = call_deepseek(
    messages=test_messages,
    temperature=0.0,
    max_tokens=30
)

print("DeepSeek API Test")
print("=" * 50)
print("Model:", DEEPSEEK_MODEL)
print("Generated Response:", test_response)

DeepSeek API Test
Model: deepseek-chat
Generated Response: Assignments


### Step 7. Compare zero-shot and few-shot results

**Your task:** Complete `compare_classification_prompts(question)`.

- Call your zero-shot prompt function.
- Call your few-shot prompt function.
- Send each prompt to your `call_deepseek` function using the same low temperature.
- Return a dictionary containing `question`, `zero_shot`, and `few_shot`.

**Discussion:** Were the two results identical? If not, why might examples change the result?

**Note:** This step sends two API requests and may incur a small cost.

**Purpose of Code:** This function runs the same student question through zero-shot and few-shot prompts. Its output helps you compare their classifications and understand how prompt design affects LLM behavior.


In [15]:
def compare_classification_prompts(question: str) -> dict:
    """Return outputs from zero-shot and few-shot classification."""
    # TODO 1: Build both prompts.
    # TODO 2: Call DeepSeek separately for each prompt.
    # TODO 3: Return {'question': ..., 'zero_shot': ..., 'few_shot': ...}.
    # ----------------------DONE----------------------

    """
    Compare zero-shot and few-shot classification outputs for the same
    student question using the DeepSeek API.

    Parameters:
        question (str): Student's question to classify.

    Returns:
        dict: Dictionary with keys 'question', 'zero_shot', and 'few_shot'.
    """
    zero_shot_prompt = build_zero_shot_prompt(question)
    few_shot_prompt = build_few_shot_prompt(question)

    system_message = {
        "role": "system",
        "content": (
            "You are a student helpdesk classification assistant for "
            "Northstar Learning Hub. Return only the allowed category name."
        ),
    }

    zero_shot_response = call_deepseek(
        messages=[
            system_message,
            {"role": "user", "content": zero_shot_prompt},
        ],
        temperature=0.1,
        max_tokens=30,
    )

    few_shot_response = call_deepseek(
        messages=[
            system_message,
            {"role": "user", "content": few_shot_prompt},
        ],
        temperature=0.1,
        max_tokens=30,
    )

    return {
        "question": question,
        "zero_shot": zero_shot_response.strip(),
        "few_shot": few_shot_response.strip(),
    }

In [16]:
sample_question = "Can I submit my homework late?"

comparison_result = compare_classification_prompts(sample_question)

print("CLASSIFICATION COMPARISON")
print("=" * 60)

print("Student Question:")
print(comparison_result["question"])

print("\nZero-Shot Classification:")
print(comparison_result["zero_shot"])

print("\nFew-Shot Classification:")
print(comparison_result["few_shot"])

print("\nComparison:")
if (
    comparison_result["zero_shot"].strip().lower()
    == comparison_result["few_shot"].strip().lower()
):
    print("Both approaches generated identical classifications.")
else:
    print("The two approaches generated different responses.")

CLASSIFICATION COMPARISON
Student Question:
Can I submit my homework late?

Zero-Shot Classification:
Assignments

Few-Shot Classification:
Assignments

Comparison:
Both approaches generated identical classifications.


In [17]:
# Validate required dictionary keys and allowed category labels
allowed_categories = {"Enrollment", "Assignments", "Certificates", "Support", "Other"}
assert set(comparison_result.keys()) == {"question", "zero_shot", "few_shot"}
assert comparison_result["zero_shot"] in allowed_categories
assert comparison_result["few_shot"] in allowed_categories

## Part D — Retrieve Relevant Context
### Step 8. Find the most relevant PDF chunk

**Your task:** Complete `retrieve_best_chunk(question, chunks)`.

Use **basic keyword overlap**, as demonstrated in the RAG lesson:
1. Convert the question to lowercase words using `re.findall(r"[a-zA-Z]+", ...)`.
2. Convert each chunk to lowercase words using the same method.
3. Score each chunk by the number of **shared unique words**.
4. Return the best chunk and its integer score as `(context, score)`.
5. If there are no chunks or every chunk has score `0`, return `("", 0)`.

**Do not use:** embeddings, vector databases, semantic-search packages, or the model to retrieve text.

**Why return a score?** It lets the caller notice obvious retrieval failures. A positive overlap score does **not** guarantee that the chunk contains the answer.

**Purpose of Code:** This function searches the handbook chunks for words that overlap with the student’s question. It selects useful context for the LLM without requiring embeddings or a vector database, and reports when no words match.


In [20]:
def retrieve_best_chunk(question: str, chunks: list[str]) -> tuple[str, int]:
    """Return the highest-overlap chunk and its keyword-overlap score."""
    # TODO 1: Convert the question to a set of lowercase words.
    # TODO 2: Compare it with words from every chunk.
    # TODO 3: Return the chunk with highest score, or ('', 0) if none match.
    # ----------------------DONE----------------------


    """
    Return the handbook chunk with the highest
    keyword-overlap score.

    Parameters:
        question (str): Student's question.
        chunks (list[str]): List of handbook text chunks.

    Returns:
        tuple[str, int]: Best matching chunk and its score.
                         Returns ("", 0) if no match exists.
    """

    if not chunks or not question.strip():
        return ("", 0)

    question_words = set(re.findall(r"[a-zA-Z]+", question.lower()))

    if not question_words:
        return ("", 0)

    best_chunk = ""
    best_score = 0

    for chunk in chunks:
        chunk_words = set(re.findall(r"[a-zA-Z]+", chunk.lower()))

        common_words = question_words.intersection(chunk_words)
        score = len(common_words)

        if score > best_score:
            best_score = score
            best_chunk = chunk


    return (best_chunk, best_score)

In [21]:
test_question = (
    "What is the deadline for each weekly assignment?"
)

retrieved_context, retrieval_score = retrieve_best_chunk(
    test_question,
    handbook_chunks
)

print("HANDBOOK RETRIEVAL TEST")
print("=" * 60)

print("Student Question:")
print(test_question)

print("\nRetrieval Score:")
print(retrieval_score)

print("\nRetrieved Context:")
if retrieved_context:
    print(retrieved_context)
else:
    print("No matching context found.")

HANDBOOK RETRIEVAL TEST
Student Question:
What is the deadline for each weekly assignment?

Retrieval Score:
7

Retrieved Context:
ilable for
revision but do not count as live attendance.
4. Weekly Assignment Deadlines
Assignments for each teaching week are due by Sunday at 11:59 PM IST. A learner may request an
extension of up to 24 hours by email before the deadline. The instructor must approve the extension in
writing. Submissions without approval after the deadline are marked late.
5. Resubmissions
A learner may resubmit each graded assignment once, within seven calendar days after receiving
feedback. The resubmission replaces the original for assessment. Additional resubmissions are not
permitted under this handbook.
6. Completion Certificate
To receive a course completion certificate, learners must satisfy all three conditions: at least 80% live
attendance, submission of every mandatory assignment, and a score of at least 60% on the final
assessment. Meeting only one or two condi

## Part E — Create the RAG Chain Using LangChain
### Step 9. Build the grounded prompt template

**Your task:** Complete `create_rag_prompt()` using `PromptTemplate`.

Your template must contain the variables `{context}` and `{question}` and tell the model:
- Answer **only** from the supplied handbook context.
- If the answer is not supported, respond **exactly**: `Insufficient context.`
- Give a short, student-friendly response.
- Do not invent policy details or assume any missing information.

**Expected output:** A LangChain `PromptTemplate` object.

**Purpose of Code:** This function defines the instructions and placeholders for the final RAG question-answering prompt. It tells the LLM to use the retrieved handbook context and avoid inventing answers when the context is insufficient.


In [23]:
def create_rag_prompt() -> PromptTemplate:
    """Create and return a grounded RAG prompt template."""
    # TODO 1: Write a template with {context} and {question}.
    # TODO 2: Specify how to handle missing information.
    # TODO 3: Return a PromptTemplate object.
    # ----------------------DONE----------------------


    """
    Create and return a grounded RAG prompt template.

    The prompt instructs the LLM to answer student
    questions using only the supplied handbook context.
    """

    template = """
        You are an AI Student Helpdesk Assistant for
        Northstar Learning Hub.

        Your task is to answer student questions using
        the information provided in the handbook context below.

        HANDBOOK CONTEXT:
        {context}

        STUDENT QUESTION:
        {question}

        INSTRUCTIONS:

        1. Answer the student's question using ONLY the
          information explicitly available in the handbook context.

        2. Do not use external knowledge, assumptions,
          or information from your general training.

        3. If the handbook context does not contain sufficient
          information to answer the question, respond with
          exactly the following:

        Insufficient context.

        4. Do not invent deadlines, policies, eligibility
          requirements, fees, benefits, or other details.

        5. Keep your response short, clear, and student-friendly.

        6. If the context partially addresses the question
          but does not provide enough information for a
          reliable answer, respond with:

        Insufficient context.

        FINAL ANSWER:
        """

    prompt = PromptTemplate(
        input_variables=["context", "question"],
        template=template
    )


    return prompt

In [24]:
rag_prompt = create_rag_prompt()

print("RAG PROMPT TEMPLATE")
print("=" * 60)

print(rag_prompt.template)

print("\nInput Variables:")
print(rag_prompt.input_variables)

print("\nPrompt Type:")
print(type(rag_prompt))

RAG PROMPT TEMPLATE

        You are an AI Student Helpdesk Assistant for
        Northstar Learning Hub.

        Your task is to answer student questions using
        the information provided in the handbook context below.

        HANDBOOK CONTEXT:
        {context}

        STUDENT QUESTION:
        {question}

        INSTRUCTIONS:

        1. Answer the student's question using ONLY the
          information explicitly available in the handbook context.

        2. Do not use external knowledge, assumptions,
          or information from your general training.

        3. If the handbook context does not contain sufficient
          information to answer the question, respond with
          exactly the following:

        Insufficient context.

        4. Do not invent deadlines, policies, eligibility
          requirements, fees, benefits, or other details.

        5. Keep your response short, clear, and student-friendly.

        6. If the context partially addresses the ques

In [25]:
sample_context = """
Students must maintain the minimum attendance
requirement specified in the handbook to qualify
for a completion certificate.
"""

sample_question = (
    "What attendance is required for a certificate?"
)

formatted_prompt = rag_prompt.format(
    context=sample_context,
    question=sample_question
)

print("FORMATTED RAG PROMPT")
print("=" * 60)
print(formatted_prompt)

FORMATTED RAG PROMPT

        You are an AI Student Helpdesk Assistant for
        Northstar Learning Hub.

        Your task is to answer student questions using
        the information provided in the handbook context below.

        HANDBOOK CONTEXT:
        
Students must maintain the minimum attendance
requirement specified in the handbook to qualify
for a completion certificate.


        STUDENT QUESTION:
        What attendance is required for a certificate?

        INSTRUCTIONS:

        1. Answer the student's question using ONLY the
          information explicitly available in the handbook context.

        2. Do not use external knowledge, assumptions,
          or information from your general training.

        3. If the handbook context does not contain sufficient
          information to answer the question, respond with
          exactly the following:

        Insufficient context.

        4. Do not invent deadlines, policies, eligibility
          requirements, fe

### Step 10. Convert a LangChain prompt into a DeepSeek API response

**Your task:** Complete `deepseek_model_step(prompt_value)`.

- LangChain passes a prompt value to your model step.
- Convert it into text using `.to_string()`.
- Pass that text into `call_deepseek` as a user message, together with a concise system message.
- Return the generated text.

**Hint:** In the next step you will wrap this Python function using `RunnableLambda`.

**Expected output:** A string. Do not create a new API key or a second client.

**Purpose of Code:** This function acts as the link between LangChain and the DeepSeek API. It converts a LangChain-formatted prompt into an API request and returns the model’s answer to the chain.


In [27]:
def deepseek_model_step(prompt_value: Any) -> str:
    """Convert a LangChain prompt value into a DeepSeek text response."""
    # TODO 1: Convert prompt_value to a plain string.
    # TODO 2: Build system and user messages.
    # TODO 3: Call call_deepseek and return the response.
    # ----------------------DONE----------------------


    """
    Convert a LangChain prompt value into plain text,
    send it to DeepSeek, and return the generated response.

    Parameters:
        prompt_value (Any): LangChain prompt value.

    Returns:
        str: Generated response from DeepSeek.
    """

    prompt_text = prompt_value.to_string()

    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful and accurate AI Student "
                "Helpdesk Assistant for Northstar Learning Hub. "
                "Follow the instructions in the user's prompt "
                "carefully. Never invent information."
            )
        },
        {
            "role": "user",
            "content": prompt_text
        }
    ]

    response = call_deepseek(
        messages=messages,
        temperature=0.2,
        max_tokens=250
    )

    return response

In [28]:
# Create a sample formatted prompt
sample_prompt_value = rag_prompt.format_prompt(
    context="""
    Students must maintain the minimum attendance
    requirement specified in the handbook to qualify
    for a completion certificate.
    """,
    question="What attendance is required for a certificate?"
)

# Send the prompt to DeepSeek
sample_answer = deepseek_model_step(sample_prompt_value)

# Display the result
print("DEEPSEEK MODEL TEST")
print("=" * 60)

print("Generated Answer:")
print(sample_answer)

print("\nResponse Type:")
print(type(sample_answer))

DEEPSEEK MODEL TEST
Generated Answer:
Insufficient context.

Response Type:
<class 'str'>


### Step 11. Combine the LangChain components

**Your task:** Complete `build_rag_chain(prompt)`.

Connect three components in this order:

```text
PromptTemplate -> RunnableLambda(deepseek_model_step) -> StrOutputParser
```

- Wrap `deepseek_model_step` in a `RunnableLambda`.
- Create a `StrOutputParser`.
- Use the LangChain pipe operator (`|`) to connect the components.
- Return the assembled chain.

**Expected output:** An object that supports `.invoke({'context': '...', 'question': '...'})`.

**Purpose of Code:** This function connects prompt preparation, the DeepSeek call, and output parsing into one reusable LangChain pipeline. The completed chain can answer different questions by receiving new context and question values.


In [30]:
def build_rag_chain(prompt: PromptTemplate):
    """Return PromptTemplate -> model runnable -> text parser chain."""
    # TODO 1: Create RunnableLambda(deepseek_model_step).
    # TODO 2: Create StrOutputParser().
    # TODO 3: Connect all three components with | and return the chain.
    # ----------------------DONE----------------------


    """
    Build a reusable LangChain RAG pipeline.

    Pipeline:
    PromptTemplate -> RunnableLambda -> StrOutputParser

    Parameters:
        prompt (PromptTemplate): Grounded RAG prompt template.

    Returns:
        Runnable: Complete LangChain RAG chain.
    """

    model_runnable = RunnableLambda(deepseek_model_step)
    output_parser = StrOutputParser()

    rag_chain = (prompt | model_runnable | output_parser)

    return rag_chain

In [31]:
# Create the RAG prompt
rag_prompt = create_rag_prompt()

# Build the complete RAG chain
rag_chain = build_rag_chain(rag_prompt)

print("RAG CHAIN CREATED SUCCESSFULLY!")
print("=" * 60)

print("Prompt Component:", type(rag_prompt))
print("Model Component:", type(RunnableLambda(deepseek_model_step)))
print("Output Parser:", type(StrOutputParser()))
print("Final Chain:", type(rag_chain))

RAG CHAIN CREATED SUCCESSFULLY!
Prompt Component: <class 'langchain_core.prompts.prompt.PromptTemplate'>
Model Component: <class 'langchain_core.runnables.base.RunnableLambda'>
Output Parser: <class 'langchain_core.output_parsers.string.StrOutputParser'>
Final Chain: <class 'langchain_core.runnables.base.RunnableSequence'>


In [32]:
test_context = """
Students must maintain the minimum attendance
requirement specified in the handbook to qualify
for a completion certificate.
"""

test_question = (
    "What attendance is required for a certificate?"
)

test_result = rag_chain.invoke({
    "context": test_context,
    "question": test_question
})

print("RAG PIPELINE TEST")
print("=" * 60)

print("Student Question:")
print(test_question)

print("\nGenerated Answer:")
print(test_result)

print("\nOutput Type:")
print(type(test_result))

RAG PIPELINE TEST
Student Question:
What attendance is required for a certificate?

Generated Answer:
Insufficient context.

Output Type:
<class 'str'>


## Part F — Connect and Test the Assistant
### Step 12. Answer one student question

**Your task:** Complete `answer_student_question(question, chunks, chain)`.

- Call `retrieve_best_chunk`.
- If the best score is `0`, return `Insufficient context.` **without** an LLM request.
- Otherwise call `chain.invoke` with `{'context': context, 'question': question}`.
- Return this dictionary:

```python
{
    'question': '...',
    'answer': '...',
    'retrieval_score': 0,
    'retrieved_context': '...'
}
```

**Important:** Even with a positive retrieval score, the model must say `Insufficient context.` when the retrieved chunk does not actually answer the question. Do not treat keyword matches as proof.

**Purpose of Code:** This function combines retrieval and answer generation into the student helpdesk workflow. It searches the handbook, skips the LLM call when no chunk matches, and returns an answer with information about the retrieved context.


In [35]:
def answer_student_question(question: str, chunks: list[str], chain) -> dict:
    """Retrieve handbook context, run the chain, and return a result dict."""
    # TODO 1: Retrieve the best context and overlap score.
    # TODO 2: Return insufficient context if retrieval failed.
    # TODO 3: Otherwise invoke the chain using context and question.
    # TODO 4: Return the required result dictionary.
    # ----------------------DONE----------------------


    """
    Retrieve relevant handbook context and generate
    a grounded answer using the RAG chain.

    Parameters:
        question (str): Student's question.
        chunks (list[str]): Handbook text chunks.
        chain: Executable LangChain RAG chain.

    Returns:
        dict: Question, answer, retrieval score,
              and retrieved handbook context.
    """

    context, retrieval_score = retrieve_best_chunk(question, chunks)

    if retrieval_score == 0:
        answer = "Insufficient context."

    else:
        answer = chain.invoke({
            "context": context,
            "question": question
        })

    result = {
        "question": question,
        "answer": answer,
        "retrieval_score": retrieval_score,
        "retrieved_context": context
    }

    return result

In [36]:
test_question = (
    "What is the minimum live attendance "
    "required for a certificate?"
)

result = answer_student_question(
    question=test_question,
    chunks=handbook_chunks,
    chain=rag_chain
)

print("STUDENT HELPDESK ASSISTANT")
print("=" * 65)

print("\nQuestion:")
print(result["question"])

print("\nGenerated Answer:")
print(result["answer"])

print("\nRetrieval Score:")
print(result["retrieval_score"])

print("\nRetrieved Handbook Context:")
print(result["retrieved_context"])

STUDENT HELPDESK ASSISTANT

Question:
What is the minimum live attendance required for a certificate?

Generated Answer:
At least 80% live attendance is required to receive a course completion certificate.

Retrieval Score:
7

Retrieved Handbook Context:
ilable for
revision but do not count as live attendance.
4. Weekly Assignment Deadlines
Assignments for each teaching week are due by Sunday at 11:59 PM IST. A learner may request an
extension of up to 24 hours by email before the deadline. The instructor must approve the extension in
writing. Submissions without approval after the deadline are marked late.
5. Resubmissions
A learner may resubmit each graded assignment once, within seven calendar days after receiving
feedback. The resubmission replaces the original for assessment. Additional resubmissions are not
permitted under this handbook.
6. Completion Certificate
To receive a course completion certificate, learners must satisfy all three conditions: at least 80% live
attendance, 

In [40]:
unsupported_question = (
    "What is the orbital period of Mars?"
)

unsupported_result = answer_student_question(
    question=unsupported_question,
    chunks=handbook_chunks,
    chain=rag_chain
)

print("INSUFFICIENT CONTEXT TEST")
print("=" * 60)

print("Question:", unsupported_result["question"])
print("Answer:", unsupported_result["answer"])
print("Retrieval Score:", unsupported_result["retrieval_score"])

# The simple word-overlap retriever will match common words (what, is, the, of), resulting in a non-zero score.
# The key validation is that the RAG model successfully identifies this and outputs 'Insufficient context.'
assert unsupported_result["answer"] == "Insufficient context."
assert unsupported_result["retrieval_score"] > 0

print("\nInsufficient-context handling successful!")

INSUFFICIENT CONTEXT TEST
Question: What is the orbital period of Mars?
Answer: Insufficient context.
Retrieval Score: 3

Insufficient-context handling successful!


### Step 14. Evaluate the assistant using a fixed test set

Use the provided six questions. For each response:
- Verify its claims directly against the handbook PDF.
- Label it `Supported`, `Unsupported`, or `Insufficient context` based on the **content**, not just the model's confidence.
- Write a one-sentence observation.
- Highlight at least one limitation of the simple word-overlap retriever.

**Your task:** Complete `evaluate_questions(test_questions, chunks, chain)` to collect the question and result for every test case. You can print your observations or write a Markdown results table below.

**Purpose of Code:** This function runs the assistant on a fixed set of student questions and collects its results. You will use those results to check whether answers are supported by the handbook and identify retrieval limitations.


In [46]:
TEST_QUESTIONS = [
    "What is the deadline for each weekly assignment?",
    "Can a student resubmit a graded assignment?",
    "What attendance is needed for a completion certificate?",
    "How many mentoring sessions can a learner book each month?",
    "Does the program guarantee a job?",
    "Does the academy give every student a free laptop?",
]

def evaluate_questions(test_questions: list[str], chunks: list[str], chain) -> list[dict]:
    """Return a list of assistant result dictionaries for the test questions."""
    # TODO 1: Loop through the test questions.
    # TODO 2: Call answer_student_question for each one.
    # TODO 3: Append each result to a list and return it.
    # ----------------------DONE----------------------

    """
    Evaluate the RAG assistant using a fixed set
    of student questions.

    Parameters:
        test_questions (list[str]): Questions to evaluate.
        chunks (list[str]): Handbook text chunks.
        chain: Executable LangChain RAG chain.

    Returns:
        list[dict]: Results containing each question,
                    generated answer, retrieval score,
                    and retrieved context.
    """

    evaluation_results = []

    for question in test_questions:
        result = answer_student_question(
            question=question,
            chunks=chunks,
            chain=chain
        )

        evaluation_results.append(result)


    return evaluation_results

In [47]:
evaluation_results = evaluate_questions(
    test_questions=TEST_QUESTIONS,
    chunks=handbook_chunks,
    chain=rag_chain
)

print("NORTHSTAR LEARNING HUB")
print("RAG ASSISTANT — EVALUATION REPORT")
print("=" * 75)

for index, result in enumerate(evaluation_results, start=1):

    print(f"\nTEST CASE {index}")
    print("-" * 75)

    print("Question:", result["question"])
    print("Answer:", result["answer"])
    print("Retrieval Score:", result["retrieval_score"])

    print("\nRetrieved Context:")
    print(result["retrieved_context"])

    print("=" * 75)

print("\nTotal Questions Evaluated:", len(evaluation_results))

NORTHSTAR LEARNING HUB
RAG ASSISTANT — EVALUATION REPORT

TEST CASE 1
---------------------------------------------------------------------------
Question: What is the deadline for each weekly assignment?
Answer: Assignments for each teaching week are due by Sunday at 11:59 PM IST.
Retrieval Score: 7

Retrieved Context:
ilable for
revision but do not count as live attendance.
4. Weekly Assignment Deadlines
Assignments for each teaching week are due by Sunday at 11:59 PM IST. A learner may request an
extension of up to 24 hours by email before the deadline. The instructor must approve the extension in
writing. Submissions without approval after the deadline are marked late.
5. Resubmissions
A learner may resubmit each graded assignment once, within seven calendar days after receiving
feedback. The resubmission replaces the original for assessment. Additional resubmissions are not
permitted under this handbook.
6. Completion Certificate
To receive a course completion certificate, learner

### Step 15. Record and explain your findings (Markdown answer)

**Purpose of this step:** This final step does not require new code. It documents what your completed assistant produced, checks its answers against the handbook, and explains the strengths and limitations you observed.

Fill in the table below after running your evaluation function. This final step tests your ability to **interpret results**, not just run code.

| Test question (short name) | Answer supported by handbook? | Your observation |
|---|---|---|
| Weekly deadline | _Fill in_ | _Fill in_ |
| Resubmission | _Fill in_ | _Fill in_ |
| Certificate attendance | _Fill in_ | _Fill in_ |
| Mentoring | _Fill in_ | _Fill in_ |
| Job guarantee | _Fill in_ | _Fill in_ |
| Free laptop | _Fill in_ | _Fill in_ |

**Answer these questions:**
1. Did zero-shot and few-shot prompting produce the same category? Explain briefly.
2. Why does the assistant retrieve a chunk before calling the LLM?
3. Give one example of a question that keyword-overlap retrieval might handle poorly.
4. Did your assistant invent any policy? If so, how did you improve the prompt or test?

#### **Evaluation Results**

| Test question | Answer supported? | Observations |
|---|---|---|
| Weekly deadline | Yes | Sunday, 11:59 PM IST; extensions of up to 24 hours require prior written approval. |
| Resubmission | Yes | One resubmission is allowed within seven calendar days after feedback. |
| Certificate attendance | Yes | At least 80% live attendance is required, along with the other two certificate conditions. |
| Mentoring | Yes | Up to two individual mentoring sessions per month. |
| Job guarantee | Yes | The handbook explicitly states that employment, interviews, and job placement are not guaranteed. |
| Free laptop | No | The retrieved context does not establish that students receive free laptops; the assistant correctly returns `Insufficient context.` |

#### **1. Did zero-shot and few-shot prompting produce the same category? Explain briefly.**

Zero-shot and few-shot prompting were used to classify the same student question into one of five categories: Enrollment, Assignments, Certificates, Support, or Other.

The zero-shot approach relied entirely on the classification instructions, whereas the few-shot approach also included labelled examples to demonstrate the expected classification pattern.

The outputs should be compared to determine whether both approaches produced the same category. If they did, it indicates that the question was relatively straightforward for the model. If they differed, the examples in the few-shot prompt may have influenced the model's interpretation of the question.

#### **2. Why does the assistant retrieve a chunk before calling the LLM?**

The assistant retrieves a relevant chunk from the handbook before calling the Large Language Model (LLM) so that the response is grounded in the academy's actual documentation.

The retrieval function uses keyword overlap to identify the chunk containing the greatest number of words shared with the student's question. This retrieved text is then passed to the LLM as context.

This approach reduces the possibility of hallucinations, prevents the model from relying unnecessarily on its general knowledge, and helps ensure that answers are based on the handbook rather than assumptions.

It also avoids calling the LLM when no matching keywords are found, returning "Insufficient context." instead.



#### **3. Give one example of a question that keyword-overlap retrieval might handle poorly.**

A question such as:

"How much time do I have to turn in my weekly coursework?"

might be handled poorly if the handbook uses different terminology, such as "assignment submission deadline" or "weekly submission schedule."

Although the question and the handbook may refer to the same concept, they do not necessarily share the same keywords.

Since the current retrieval method relies on exact word overlap rather than semantic similarity, it may fail to retrieve the most relevant chunk.

A more advanced retrieval system using embeddings and vector similarity could improve its ability to identify semantically related content.



#### **4. Did your assistant invent any policy? If so, how did you improve the prompt or test?**

The assistant was designed to minimize policy-related hallucinations by restricting its responses to information explicitly available in the retrieved handbook context.

The RAG prompt instructs the model not to use external knowledge, assumptions, or unsupported information. It also specifies that the exact response "Insufficient context." must be returned whenever the supplied context does not contain enough information to answer reliably.

Additionally, the evaluation includes questions about potentially unsupported policies, such as job guarantees and free laptops. These questions help test whether the assistant can distinguish documented information from assumptions.

The prompt was strengthened with explicit grounding instructions, and the evaluation process checks whether the generated answers are supported by the retrieved handbook content.

Overall, the project demonstrates how prompt engineering, keyword-based retrieval, and a Retrieval-Augmented Generation (RAG) pipeline can be combined to build a basic AI-powered student helpdesk assistant.

## Submission Checklist


Upload your completed `.ipynb` with executed outputs on GitHub and share the link of repository.



- [ ] All 12 unfinished Python functions have been implemented (Steps 2–12 and Step 14).
- [ ] The notebook runs sequentially from a fresh Colab runtime after adding the API key and uploading the PDF.
- [ ] The zero-shot/few-shot comparison has an output.
- [ ] The PDF has been extracted and chunked successfully.
- [ ] A LangChain pipeline produces a grounded answer.
- [ ] The assistant handles a question that is not answered in the PDF.
- [ ] All six evaluation questions and the reflection are completed.
- [ ] The DeepSeek API key is not embedded in code or visible in notebook output.

### Assessment Rubric

| Criterion | Marks |
|---|---:|
| Zero-shot and few-shot prompt quality | 20 |
| Direct LLM API integration | 20 |
| LangChain chain construction | 20 |
| PDF processing, retrieval, grounded answers | 25 |
| Testing, observations, and submission quality | 15 |
| **Total** | **100** |

### Helpful references
- DeepSeek API: https://api-docs.deepseek.com/
- LangChain: https://python.langchain.com/
- PyPDF: https://pypdf.readthedocs.io/
